# Quasi-steady simulation of a test-bench log

This notebook takes a long time-series log from a test bench
(`monocell_datas_03Mar2026_08h10.csv`, MEA62) and, **without aggregating
into discrete operating points**, replays the logged setpoints through
a `FuelCell` model:

1. Load the log.
2. Build the MEA62 cell (same 18-parameter estimation result as in
   `02_parameter_estimation.ipynb`).
3. For (almost) every logged sample, build a `CellConditions` from the
   logged setpoints (current, cell temperature, pressures, relative
   humidities, stoichiometries) and evaluate the steady-state model via
   `ExplicitSteadyStateModel.solve`.
4. Plot the simulated and measured cell voltage against time.

**Caveat.** The model is evaluated quasi-statically: each sample is treated
as an independent steady-state operating point, ignoring thermal and water
transients between samples. Periods where `I_Pile(A)` is (close to) zero
are excluded because the steady-state model is not well-defined at zero
current. The number of cells in the stack (`N_CELLS`) and the active area
(`CELL_AREA`) are not recorded in the log; they are exposed below as
parameters to adjust for the actual test rig.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import marapendi as mrpd

## 1 — Load the log

In [ ]:
df = pd.read_csv('monocell_datas_03Mar2026_08h10.csv', sep=';', skiprows=6, encoding='latin1')
df = df.rename(columns=lambda c: c.strip())

# The header row has one fewer name than the data has columns: the first
# (unnamed) data column -- which pandas turns into the index -- holds the
# real `Time(s)`, and every named column is shifted one position to the
# left of its data. Re-align the column names accordingly.
old_columns = list(df.columns)
df = df.reset_index()
df.columns = ['Time(s)'] + old_columns[1:] + ['_extra']
df = df.drop(columns=['_extra'])

df['Time(h)'] = (df['Time(s)'] - df['Time(s)'].iloc[0]) / 3600.
df[['Time(h)', 'I_Pile(A)', 'U_Pile(V)', 'T_pile(°C)']].describe()

## 2 — Cell parameters and assembly

Same 18-parameter MEA62 estimation result (condition 2, left-out condition 1)
as in `02_parameter_estimation.ipynb`.

The membrane now uses the new composition API: ionomer properties are passed
inside a `PFSAIonomer` instance, and `PFSA` only owns the membrane-level
geometry and permeation model.

In [ ]:
# Fixed parameters.
fixed_parameters = {
    'radius-carbon': 25e-9,
    'ionomer-E-act-cond': 15e6,
    'n_s': 2,
    'ionomer-k1': 8.5,
    'ionomer-k2': 5.4,
    'ionomer-k3': 5.4,
    'gdl-porosity': 0.6,
    'pt-wt-percent': 0.4,
    'ch-height': 1e-3,
    'gdl-thickness': 150e-6,
    'gdl-theta': 120.,
    'gdl-eff-diff-ratio': 0.3,
    'cl-abs-perm': 1e-13,
    'wet-transition': 0.4,
    'pt-loading': .3e-2,
    'ic-ratio': 1.4,
    'ecsa': 60e3,
    'memb-thickness': 12e-6,
    'memb-water-diff': 2e-10,
    'E-act-memb-diff': 20e6,
    'E-act-memb-abs': 20e6,
    'cl-theta': 97.,
    'cl-thermal-cond': 0.22,
    'cl-pore-diameter': 40e-9,
}

# 18 parameters estimated for MEA62, condition 1 left out,
# from results/results_final_estimation_model2_new_perm_lim_cv.csv
# (n_parameters=18, test_case=2).
estimated_parameters = {
    'elec-resistance': 3.2018410582982336e-06,
    'alpha-c': 0.8804552030152384,
    'memb-cond-correction': 10.194306339919532,
    'B_ch': 1.3173241932454605,
    'ionomer-cond-corr': 0.16788866561668214,
    'i0-c': 0.0013603559102389256,
    'memb-cond-exp': 1.6472232706926844,
    'Sh': 0.7956740630180096,
    'E-act-ca': 73404895.12308666,
    'memb-equiv-weight': 707.0461410229138,
    'memb-E-act-cond': 12920411.386859203,
    'gdl-thermal-cond': 0.10151383504290674,
    'gamma-c': 0.7815865333197847,
    'memb-abs-constant': 3.680688030527334e-05,
    'ix-corr': 2.0,
    'ionomer-cond-exp': 1.0,
    'tcr': 0.0009955086394233985,
    'gdl-abs-perm': 9.999999010000095e-12,
}

params = {**fixed_parameters, **estimated_parameters}


def create_cell(params):
    # Custom H2 permeation model (Goshtasbi et al., 2020) — subclasses
    # HydrogenPermeationModel so it slots directly into PFSA.h2_permeation_model.
    class NewPermModel(mrpd.HydrogenPermeationModel):
        def permeation_flux(self, membrane_thickness, partial_pressure_h2,
                            temperature, pressure_difference, water_vol_fraction):
            return self.permeability_correction_factor * (
                15.7e-9 * np.exp(-20280 / 8.3415 / temperature) +
                water_vol_fraction * 45e-9 * np.exp(-18930 / 8.3145 / temperature)
            ) / 1000 * 100 / 1e5 * partial_pressure_h2 / membrane_thickness

    # Ionomer material properties go into PFSAIonomer; PFSA owns geometry + permeation.
    membrane = mrpd.PFSA(
        ionomer=mrpd.PFSAIonomer(
            equivalent_weight=params['memb-equiv-weight'],
            dry_density=2000.,
            conductivity_exp=params['memb-cond-exp'],
            conductivity_activation_energy=params['memb-E-act-cond'],
            conductivity_correction=params['memb-cond-correction'],
            reference_water_diffusivity=params['memb-water-diff'],
            reference_water_absorption_coefficient=params['memb-abs-constant'],
            water_diffusivity_activation_energy=params['E-act-memb-diff'],
            water_absorption_activation_energy=params['E-act-memb-abs'],
        ),
        dry_thickness=params['memb-thickness'],
        h2_permeation_model=NewPermModel(permeability_correction_factor=params['ix-corr']),
    )

    orr_kinetics = mrpd.ElectrochemicalReaction(
        reference_exchange_current_density=params['i0-c'],
        reaction_order=params['gamma-c'],
        activation_energy=params['E-act-ca'],
        reference_activity=1.01325e5,
        reference_temperature=353.15,
        number_of_electrons=1,
        charge_transfer_coeff=params['alpha-c'],
    )

    liq_model = mrpd.DarcyTransportModel(J_function_exponent=params['wet-transition'])

    gdl = {
        side: mrpd.GasDiffusionLayer(
            thickness=params['gdl-thickness'],
            contact_angle=params['gdl-theta'],
            effective_gas_diffusion_ratio=params['gdl-eff-diff-ratio'],
            absolute_permeability=params['gdl-abs-perm'],
            porosity=params['gdl-porosity'],
            thermal_conductivity=params['gdl-thermal-cond'],
            two_phase_transport_model=liq_model,
            transport_resistance_model=mrpd.PorousGasDiffusionModel(
                water_saturation_exponent=params['n_s']),
        ) for side in ['ca', 'an']
    }

    ch = {
        side: mrpd.FlowChannel(
            height=params['ch-height'], width=1e-3, n_parallel=1, length=21 * 50e-3,
            reactant='o2' if side == 'ca' else 'h2',
            transport_resistance_model=mrpd.ChannelGasResistanceModel(
                sherwood=params['Sh'], B_ch=params['B_ch']),
        ) for side in ['an', 'ca']
    }

    ionomer = mrpd.PFSAIonomer(
        conductivity_correction=params['ionomer-cond-corr'],
        conductivity_exp=params['ionomer-cond-exp'],
        conductivity_activation_energy=params['ionomer-E-act-cond'],
    )

    ca_cl = mrpd.PtCCatalystLayer(
        ecsa=params['ecsa'], platinum_loading=params['pt-loading'], ionomer=ionomer,
        catalyst_platinum_weight_percent=params['pt-wt-percent'],
        ionomer_to_carbon_ratio=params['ic-ratio'],
        ionomer_k1=params['ionomer-k1'], ionomer_k2=params['ionomer-k2'],
        ionomer_k3=params['ionomer-k3'],
        pore_diameter=params['cl-pore-diameter'], omega_PtO=0,
        carbon_agglomerate_radius=params['radius-carbon'],
        thickness=params['pt-loading'] * 2.8e-6 / 0.1e-2,
        absolute_permeability=params['cl-abs-perm'], contact_angle=params['cl-theta'],
        thermal_conductivity=params['cl-thermal-cond'], reaction=orr_kinetics,
        two_phase_transport_model=liq_model,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(water_saturation_exponent=1.5),
    )

    an_cl = mrpd.PtCCatalystLayer(
        ecsa=params['ecsa'], platinum_loading=1e-3,
        catalyst_platinum_weight_percent=params['pt-wt-percent'],
        ionomer_to_carbon_ratio=params['ic-ratio'], ionomer=ionomer,
        pore_diameter=params['cl-pore-diameter'],
        carbon_agglomerate_radius=params['radius-carbon'],
        thickness=2.8e-6, absolute_permeability=params['cl-abs-perm'],
        contact_angle=params['cl-theta'],
        thermal_conductivity=params['cl-thermal-cond'],
        two_phase_transport_model=liq_model,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(water_saturation_exponent=1.5),
    )

    return mrpd.FuelCell(
        electrical_resistance=params['elec-resistance'],
        area=25e-4,
        ca=mrpd.FuelCellSide(
            cl=ca_cl, gdl=gdl['ca'], has_gdl=True, has_mpl=False, ch=ch['ca'],
            thermal_contact_resistance=params['tcr'],
        ),
        an=mrpd.FuelCellSide(
            cl=an_cl, gdl=gdl['an'], has_gdl=True, has_mpl=False, ch=ch['an'],
            thermal_contact_resistance=params['tcr'],
        ),
        membrane=membrane,
    )


cell = create_cell(params)
print(f"Cell assembled — membrane thickness: {cell.membrane.dry_thickness*1e6:.0f} µm")

## 3 — Simulate the time evolution

Each sample is evaluated independently as a steady-state point via
`ExplicitSteadyStateModel.solve`. Samples with `I_Pile(A) <= I_MIN` are
skipped (left as `NaN`) because the steady-state model is not defined at
zero current. Stoichiometry values from the log are used directly; they
are well-defined when current is non-zero.

In [ ]:
CELL_AREA = 25e-4   # m², active area per cell
N_CELLS   = 1       # number of cells in series (set to actual stack size)
I_MIN     = 0.5     # A, samples below this current are excluded
T_START_H = 1.6     # h, start of polarization curve section
T_END_H   = 2.8     # h, end of polarization curve section

n = len(df)
cell_voltage = np.full(n, np.nan)
active = (
    (df['I_Pile(A)'] > I_MIN)
    & (df['Time(h)'] >= T_START_H)
    & (df['Time(h)'] <= T_END_H)
).to_numpy()

model = mrpd.ExplicitSteadyStateModel()

for k in np.flatnonzero(active):
    row = df.iloc[k]
    T = row['T_pile(°C)'] + 273.15

    cond = mrpd.CellConditions(
        current_density=np.array([max(row['I_Pile(A)'], 1e-4) / CELL_AREA]),
        cell_temperature=T,
        ca=mrpd.SideConditions(
            inlet_temperature=row['T_Air_in(°C)'] + 273.15,
            outlet_pressure=row['P_Air_Out(bara)'] * 1e5,
            dry_o2_mole_fraction=0.21,
            stoichiometry=row['Stoeckio_air_calc'],
            inlet_relative_humidity=row['RH_Air_calc(%)'] / 100.,
        ),
        an=mrpd.SideConditions(
            inlet_temperature=row['T_H2_In(°C)'] + 273.15,
            outlet_pressure=row['P_h2_out(bara)'] * 1e5,
            dry_h2_mole_fraction=1.0,
            stoichiometry=row['Stoeckio_h2_calc'],
            inlet_relative_humidity=row['RH_h2_calc(%)'] / 100.,
        ),
    )

    state = model.set_initial_conditions(cell, cond)
    state = model.solve(cell, cond, state)
    cell_voltage[k] = float(np.atleast_1d(state.cell_voltage)[0])

print(f"Evaluated {active.sum()} samples in [{T_START_H}, {T_END_H}] h")

## 4 — Compare to the log

In [ ]:
time_h = df['Time(h)'].to_numpy()
current_density = df['I_Pile(A)'].to_numpy() / CELL_AREA * 1e-4   # A/cm²
measured_cell_voltage = df['U_Pile(V)'].to_numpy() / N_CELLS

window = (time_h >= T_START_H) & (time_h <= T_END_H)
t_w   = time_h[window]

fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

axes[0].plot(t_w, current_density[window])
axes[0].set_ylabel('Current density (A/cm²)')

axes[1].plot(t_w, measured_cell_voltage[window], label=f'Measured (U_Pile / N_CELLS={N_CELLS})')
axes[1].plot(t_w, cell_voltage[window], label='Simulated (FuelCell steady-state)')
axes[1].set_ylabel('Cell voltage (V)')
axes[1].set_ylim([0.55, 0.95])
axes[1].legend()

axes[2].plot(t_w, np.abs(1e3 * (measured_cell_voltage[window] - cell_voltage[window])))
axes[2].set_ylabel('Cell voltage error (mV)')
axes[2].set_ylim([0, 30])

for ax in axes:
    ax.set_xlabel('Time (h)')
    ax.grid()

fig.suptitle('Quasi-steady simulation — MEA62 polarization curve')
fig.tight_layout()
plt.show()